# Project 1: R&D Quintiles (CAPM + FF3)

Based on `recoded_project_0.ipynb`. **R&D:** No R&D vs. five **equally populated quintiles** of firms with XRD (option: **normalized** = XRD as % of net income, or **non-normalized** = raw XRD). **Models:** CAPM and Fama–French 3-factor. Same metrics; no investability flag.

In [12]:
import wrds
import pandas as pd
import numpy as np
import statsmodels.api as sm
from statsmodels.regression.linear_model import OLS
from statsmodels.tools.tools import add_constant
import plotly.graph_objects as go
from pathlib import Path
import pickle

def connect_wharton(username="zkg232"):
    """Connect to WRDS PostgreSQL (our prior method: explicit username, uses .pgpass for password)."""
    return wrds.Connection(wrds_username=username)

db = connect_wharton()

# Cache for WRDS data to avoid re-running queries
CACHE_DIR = Path('cache')
CACHE_DIR.mkdir(exist_ok=True)

def load_or_fetch(name, loader):
    path = CACHE_DIR / f"{name}.pkl"
    if path.exists():
        with open(path, 'rb') as f:
            return pickle.load(f)
    data = loader()
    with open(path, 'wb') as f:
        pickle.dump(data, f)
    return data

Loading library list...
Done


## Compustat Query - Get R&D Data and Link to CRSP

In [13]:
comp_sql = """
SELECT
    a.gvkey, 
    a.datadate, 
    a.xrd,
    a.ni,
    b.lpermno AS permno
FROM
    comp.funda a
JOIN
    crsp.ccmxpf_linktable b ON a.gvkey = b.gvkey
WHERE
    a.indfmt = 'INDL'
    AND a.datafmt = 'STD'
    AND a.popsrc = 'D'
    AND a.consol = 'C'
    AND a.fyear BETWEEN 1978 AND 2022
    AND a.fic = 'USA'
    AND a.curcd = 'USD'
    AND a.exchg BETWEEN 11 AND 19
    AND (a.sich < 6000 OR a.sich > 6999 OR a.sich IS NULL)
    AND (a.sich != 2834 OR a.sich IS NULL)
    AND b.linktype IN ('LU', 'LC')
    AND b.linkprim IN ('P', 'C')
    AND b.usedflag = 1
    AND (b.linkdt <= a.datadate OR b.linkdt IS NULL)
    AND (b.linkenddt >= a.datadate OR b.linkenddt IS NULL)
"""
comp = load_or_fetch('comp_quintiles', lambda: db.raw_sql(comp_sql))

## CRSP Query - Monthly Returns with Delisting Adjustment

In [14]:
crsp_sql = """
SELECT 
    a.permno,
    a.date,
    a.ret,
    a.prc,
    a.shrout,
    c.dlret
FROM 
    crsp.msf a
INNER JOIN crsp.msenames b ON a.permno = b.permno
    AND a.date BETWEEN b.namedt AND b.nameendt
LEFT JOIN crsp.msedelist c ON a.permno = c.permno
    AND date_trunc('month', a.date) = date_trunc('month', c.dlstdt)
WHERE 
    a.date BETWEEN '1970-01-01' AND '2023-06-30'
    AND a.ret IS NOT NULL
    AND a.ret >= -1
    AND b.shrcd IN (10, 11)
    AND (b.siccd < 6000 OR b.siccd > 6999 OR b.siccd IS NULL)
    AND (b.siccd != 2834 OR b.siccd IS NULL)
    AND b.exchcd IN (1, 2, 3)
"""
crsp = load_or_fetch('crsp', lambda: db.raw_sql(crsp_sql))

## Adjust for Delisting Returns

In [15]:
crsp['dlret'] = pd.to_numeric(crsp['dlret'], errors='coerce')
crsp['ret'] = np.where(crsp['dlret'].notna(),
                       (1 + crsp['ret']) * (1 + crsp['dlret']) - 1,
                       crsp['ret'])

## Market Return and Risk-Free Rate

In [16]:
mkt = load_or_fetch('mkt', lambda: db.raw_sql("""
    SELECT date, vwretd, ewretd
    FROM crsp.msi
    WHERE date BETWEEN '1970-01-01' AND '2023-06-30'
"""))

ff = load_or_fetch('ff_factors', lambda: db.raw_sql("""
    SELECT date, rf, mktrf, smb, hml
    FROM ff.factors_monthly
    WHERE date BETWEEN '1970-01-01' AND '2023-06-30'
"""))

## R&D Measure: Lag and (Option) Normalize

In [17]:
comp['xrd'] = pd.to_numeric(comp['xrd'], errors='coerce')
comp['ni'] = pd.to_numeric(comp['ni'], errors='coerce')
comp = comp.sort_values(['permno', 'datadate'])
comp['xrd_lag1'] = comp.groupby('permno')['xrd'].shift(1)
comp['ni_lag1'] = comp.groupby('permno')['ni'].shift(1)
# Normalized: XRD as % of net income (only when ni_lag1 > 0)
comp['xrd_pct_ni_lag1'] = np.where(comp['ni_lag1'].notna() & (comp['ni_lag1'] > 0),
                                    comp['xrd_lag1'] / comp['ni_lag1'], np.nan)

## 4-month (120-day) reporting lag — available_date

In [18]:
# Monthly formation: use fiscal data only from available_date = datadate + 120 days (4 months)
comp['datadate'] = pd.to_datetime(comp['datadate'])
comp['available_date'] = comp['datadate'] + pd.Timedelta(days=120)
comp['available_date'] = comp['available_date'].dt.to_period('M').dt.to_timestamp()  # floor to month for joins
comp = comp.drop_duplicates(subset=['permno', 'datadate'], keep='last')

crsp['date'] = pd.to_datetime(crsp['date'])
crsp['date'] = crsp['date'].dt.to_period('M').dt.to_timestamp()  # floor to month for joins

## Market Cap - Use Lagged for Value Weighting

In [19]:
crsp['ret'] = pd.to_numeric(crsp['ret'], errors='coerce')
crsp['prc'] = pd.to_numeric(crsp['prc'], errors='coerce')
crsp['shrout'] = pd.to_numeric(crsp['shrout'], errors='coerce')
crsp['mktcap'] = crsp['prc'].abs() * crsp['shrout']
crsp = crsp.sort_values(['permno', 'date'])
crsp['mktcap_lag'] = crsp.groupby('permno')['mktcap'].shift(1)

## Merge Datasets (monthly formation: reprex-style merge + date filter + keep last)

In [20]:
# Reprex-style merge: pass lagged R&D so we can form quintiles in analysis
comp_for_merge = comp[['permno', 'available_date', 'xrd_lag1', 'ni_lag1', 'xrd_pct_ni_lag1']].copy()
# Keep all rows (NaN xrd_lag1 → No R&D in quintile assignment)
comp_for_merge['permno'] = pd.to_numeric(comp_for_merge['permno'], errors='coerce').astype('int64')
comp_for_merge = comp_for_merge.dropna(subset=['permno'])
comp_for_merge['available_date'] = pd.to_datetime(comp_for_merge['available_date']).dt.to_period('M').dt.to_timestamp()

crsp_dates = crsp[['permno', 'date']].copy()
crsp_dates['permno'] = pd.to_numeric(crsp_dates['permno'], errors='coerce').astype('int64')
crsp_dates['date'] = pd.to_datetime(crsp_dates['date']).dt.to_period('M').dt.to_timestamp()
crsp_dates = crsp_dates.dropna().drop_duplicates()

merged = crsp_dates.merge(comp_for_merge, on='permno', how='inner')
merged = merged[merged['date'] >= merged['available_date']]
merged = merged.sort_values(['permno', 'date', 'available_date'])
merged = merged.drop_duplicates(subset=['permno', 'date'], keep='last')

crsp_cols = ['permno', 'date', 'ret', 'prc', 'mktcap', 'mktcap_lag']
merged = merged.merge(crsp[crsp_cols], on=['permno', 'date'], how='inner')
merged = merged[(merged['available_date'].dt.year >= 1980) & (merged['available_date'].dt.year <= 2022)]
merged = merged.dropna(subset=['ret', 'mktcap_lag'])
merged = merged[merged['mktcap_lag'] > 0]

## Analysis Data: R&D Quintiles (No R&D + Q1–Q5, equal count)

In [21]:

def assign_quintiles_per_date(df):
    out = []
    for date, g in df.groupby('date'):
        no_rd = g[~g['has_rd']].copy()
        no_rd['rd_quintile'] = 'No R&D'
        with_rd = g[g['has_rd']].copy()
        if len(with_rd) < 5:
            with_rd['rd_quintile'] = 'Q1'
        else:
            with_rd['rd_quintile'] = pd.qcut(with_rd['rd_measure'].rank(method='first'), 5, labels=['Q1', 'Q2', 'Q3', 'Q4', 'Q5']).astype(str)
        out.append(pd.concat([no_rd, with_rd], ignore_index=True))
    return pd.concat(out, ignore_index=True)

def build_analysis_data(merged, use_normalized):
    """Build analysis_data with quintiles. use_normalized: True = XRD % of NI, False = raw XRD."""
    ad = merged[['permno', 'date', 'ret', 'prc', 'mktcap', 'mktcap_lag', 'xrd_lag1', 'ni_lag1', 'xrd_pct_ni_lag1']].copy()
    if use_normalized:
        ad['rd_measure'] = ad['xrd_pct_ni_lag1']
        ad['has_rd'] = ad['rd_measure'].notna() & (ad['rd_measure'] > 0)
    else:
        ad['rd_measure'] = ad['xrd_lag1']
        ad['has_rd'] = ad['rd_measure'].notna() & (ad['rd_measure'] > 0)
    ad = ad.sort_values(['date', 'permno'])
    ad = assign_quintiles_per_date(ad)
    ad['portfolio_id'] = ad['rd_quintile']
    return ad

# Run for both variants (full functional flow)
analysis_data_norm = build_analysis_data(merged, use_normalized=True)
analysis_data_nonnorm = build_analysis_data(merged, use_normalized=False)


## Portfolio Returns, CAPM, FF3, Tables, Chart

In [22]:

def format_numeric_cols(data, rounding_map):
    data = data.copy()
    for col, digits in rounding_map.items():
        if col in data.columns and pd.api.types.is_numeric_dtype(data[col]):
            data[col] = data[col].round(digits)
    return data

def calculate_portfolio_returns(data, group_var='research_not'):
    ret_col = 'ret' if 'ret' in data.columns else 'ret_crsp'
    prc_col = 'prc' if 'prc' in data.columns else 'prc_crsp'
    data = data[data[ret_col].notna() & data[prc_col].notna() & data['mktcap'].notna()].copy()
    data = data.sort_values(['permno', 'date'])
    if 'mktcap_lag' not in data.columns:
        data['mktcap_lag'] = data.groupby('permno')['mktcap'].shift(1)
    data['prc_lag'] = data.groupby('permno')[prc_col].shift(1)
    data = data[data['mktcap_lag'].notna() & data['prc_lag'].notna()]
    data['date'] = data['date'] + pd.offsets.MonthEnd(0)
    def calc_vw_return(group):
        total_mktcap = group['mktcap_lag'].sum()
        if total_mktcap == 0:
            return np.nan
        weights = group['mktcap_lag'] / total_mktcap
        return (weights * group[ret_col]).sum()
    portfolio_returns = data.groupby(['date', group_var]).apply(
        lambda x: pd.Series({
            'vw_return': calc_vw_return(x),
            'ew_return': x[ret_col].mean(),
            'total_mktcap': x['mktcap_lag'].sum(),
            'n_stocks': len(x)
        })
    ).reset_index()
    return portfolio_returns.sort_values([group_var, 'date'])

def add_long_short_spreads(portfolio_returns, group_col='portfolio_id'):
    """Append assignment long-short portfolios: (1) Q5 - No R&D, (2) Q5 - Q1. Both EW and VW."""
    if group_col not in portfolio_returns.columns or 'ew_return' not in portfolio_returns.columns:
        return portfolio_returns
    wide_ew = portfolio_returns.pivot(index='date', columns=group_col, values='ew_return')
    wide_vw = portfolio_returns.pivot(index='date', columns=group_col, values='vw_return')
    needed = ['No R&D', 'Q1', 'Q5']
    for col in needed:
        if col not in wide_ew.columns or col not in wide_vw.columns:
            return portfolio_returns
    out = [portfolio_returns]
    for name, minus_col in [('Q5-No R&D', 'No R&D'), ('Q5-Q1', 'Q1')]:
        spread_ew = (wide_ew['Q5'] - wide_ew[minus_col]).dropna()
        spread_vw = (wide_vw['Q5'] - wide_vw[minus_col]).dropna()
        common_idx = spread_ew.index.intersection(spread_vw.index)
        spread_ew, spread_vw = spread_ew.loc[common_idx], spread_vw.loc[common_idx]
        spread_df = pd.DataFrame({
            'date': spread_ew.index,
            group_col: name,
            'ew_return': spread_ew.values,
            'vw_return': spread_vw.values,
            'total_mktcap': 0.0,
            'n_stocks': 0
        })
        out.append(spread_df)
    return pd.concat(out, ignore_index=True).sort_values([group_col, 'date'])

def run_capm_regression(portfolio_returns, market_returns, rf_rate, use_ew_benchmark=False):
    if 'rf' not in market_returns.columns or market_returns['rf'].isna().all():
        raise ValueError("Risk-free rate 'rf' must be present in market_returns (from Fama-French ff.factors_monthly).")
    market_col = 'ewretd' if use_ew_benchmark else 'vwretd'
    portfolio_returns = portfolio_returns.copy()
    portfolio_returns['date'] = portfolio_returns['date'] + pd.offsets.MonthEnd(0)
    cols = ['date', market_col, 'rf']
    market_data = market_returns[cols].copy()
    market_data['date'] = market_data['date'] + pd.offsets.MonthEnd(0)
    market_data = market_data.rename(columns={market_col: 'market_return'})
    reg_data = portfolio_returns.merge(market_data, on='date', how='left')
    reg_data = reg_data[reg_data['portfolio_return'].notna() & reg_data['market_return'].notna() & reg_data['rf'].notna()].copy()
    r = reg_data['rf'].astype(float)
    reg_data['portfolio_excess'] = reg_data['portfolio_return'] - r
    reg_data['market_excess'] = reg_data['market_return'] - r
    reg_data = reg_data[reg_data['portfolio_excess'].notna() & reg_data['market_excess'].notna()].copy()
    if len(reg_data) < 2:
        return None
    market_excess = pd.to_numeric(reg_data['market_excess'], errors='coerce')
    portfolio_excess = pd.to_numeric(reg_data['portfolio_excess'], errors='coerce')
    valid_mask = market_excess.notna() & portfolio_excess.notna()
    market_excess = market_excess[valid_mask].to_numpy(dtype=float)
    portfolio_excess = portfolio_excess[valid_mask].to_numpy(dtype=float)
    if len(market_excess) < 2:
        return None
    X = pd.DataFrame({'market_excess': market_excess})
    X = add_constant(X, has_constant='add')
    model = OLS(portfolio_excess, X).fit()
    n_months = len(portfolio_excess)
    arithmetic_monthly_return = pd.to_numeric(reg_data.loc[valid_mask, 'portfolio_return'], errors='coerce').mean()
    monthly_vol = portfolio_excess.std(ddof=1)
    rf_mean_sample = reg_data.loc[valid_mask, 'rf'].mean()
    params, tvalues, pvalues = model.params, model.tvalues, model.pvalues
    alpha = params.iloc[0]
    beta = params.iloc[1]
    alpha_tstat = tvalues.iloc[0]
    beta_tstat = tvalues.iloc[1]
    alpha_pval = pvalues.iloc[0]
    beta_pval = pvalues.iloc[1]
    ci = model.conf_int(alpha=0.05)
    return pd.Series({
        'arithmetic_monthly_return': arithmetic_monthly_return,
        'annualized_return': (1 + arithmetic_monthly_return) ** 12 - 1,
        'alpha': alpha, 'beta': beta,
        'alpha_tstat': alpha_tstat, 'beta_tstat': beta_tstat,
        'alpha_pval': alpha_pval, 'beta_pval': beta_pval,
        'alpha_lower_ci': ci.iloc[0, 0], 'alpha_upper_ci': ci.iloc[0, 1],
        'beta_lower_ci': ci.iloc[1, 0], 'beta_upper_ci': ci.iloc[1, 1],
        'r_squared': model.rsquared,
        'sharpe_ratio': (arithmetic_monthly_return - rf_mean_sample) / monthly_vol * np.sqrt(12) if monthly_vol > 0 else np.nan,
        'annualized_alpha': alpha * 12,
        'monthly_vol': monthly_vol,
        'annualized_vol': monthly_vol * np.sqrt(12),
        'n_observations': n_months
    })

def run_ff3_regression(portfolio_returns, market_returns, rf_rate):
    """Fama-French 3-factor: R - rf = alpha + b_mkt*mktrf + b_smb*smb + b_hml*hml."""
    for c in ['rf', 'mktrf', 'smb', 'hml']:
        if c not in market_returns.columns or market_returns[c].isna().all():
            raise ValueError(f"FF3 factor '{c}' must be in market_returns (from ff.factors_monthly).")
    portfolio_returns = portfolio_returns.copy()
    portfolio_returns['date'] = portfolio_returns['date'] + pd.offsets.MonthEnd(0)
    cols = ['date', 'mktrf', 'smb', 'hml', 'rf']
    market_data = market_returns[cols].copy()
    market_data['date'] = pd.to_datetime(market_data['date']) + pd.offsets.MonthEnd(0)
    reg_data = portfolio_returns.merge(market_data, on='date', how='left')
    reg_data = reg_data[reg_data['portfolio_return'].notna() & reg_data['mktrf'].notna() & reg_data['rf'].notna()].copy()
    reg_data['portfolio_excess'] = reg_data['portfolio_return'] - reg_data['rf'].astype(float)
    reg_data = reg_data[reg_data['portfolio_excess'].notna()].copy()
    if len(reg_data) < 5:
        return None
    y = reg_data['portfolio_excess'].astype(float).values
    X = reg_data[['mktrf', 'smb', 'hml']].astype(float)
    X = add_constant(X, has_constant='add')
    model = OLS(y, X).fit()
    n_months = len(y)
    arithmetic_monthly_return = reg_data['portfolio_return'].mean()
    monthly_vol = reg_data['portfolio_excess'].std(ddof=1)
    rf_mean = reg_data['rf'].mean()
    params, tvalues, pvalues, ci = model.params, model.tvalues, model.pvalues, model.conf_int(alpha=0.05)
    return pd.Series({
        'arithmetic_monthly_return': arithmetic_monthly_return,
        'annualized_return': (1 + arithmetic_monthly_return) ** 12 - 1,
        'alpha': params.iloc[0], 'beta': params.iloc[1],
        'beta_mkt': params.iloc[1], 'beta_smb': params.iloc[2], 'beta_hml': params.iloc[3],
        'alpha_tstat': tvalues.iloc[0], 'beta_tstat': tvalues.iloc[1],
        'beta_smb_tstat': tvalues.iloc[2], 'beta_hml_tstat': tvalues.iloc[3],
        'alpha_pval': pvalues.iloc[0], 'beta_pval': pvalues.iloc[1],
        'alpha_lower_ci': ci.iloc[0, 0], 'alpha_upper_ci': ci.iloc[0, 1],
        'beta_lower_ci': ci.iloc[1, 0], 'beta_upper_ci': ci.iloc[1, 1],
        'r_squared': model.rsquared,
        'sharpe_ratio': (arithmetic_monthly_return - rf_mean) / monthly_vol * np.sqrt(12) if monthly_vol > 0 else np.nan,
        'n_observations': n_months
    })

def calculate_sharpe_ratio(returns, rf_rate):
    if len(returns) == 0 or returns.isna().all():
        return np.nan
    excess = returns - rf_rate
    sd = returns.std()
    if sd == 0 or pd.isna(sd):
        return np.nan
    return (excess.mean() / sd) * np.sqrt(12)

def calculate_all_portfolio_metrics(portfolio_data, market_data, portfolio_name, rf_rate, model='capm'):
    """model: 'capm' or 'ff3'."""
    group_col = portfolio_data.columns[1]
    run_reg = (lambda pr, md, rf: run_capm_regression(pr, md, rf, use_ew_benchmark=False)) if model == 'capm' else run_ff3_regression
    ew_returns = portfolio_data[['date', 'ew_return', group_col]].copy()
    ew_returns = ew_returns.rename(columns={'ew_return': 'portfolio_return'})
    ew_metrics_list = []
    for group_name, group_data in ew_returns.groupby(group_col):
        r = run_reg(group_data[['date', 'portfolio_return']], market_data, rf_rate)
        if r is not None and len(r) > 0:
            ew_metrics_list.append({**r.to_dict(), group_col: group_name})
    ew_metrics = pd.DataFrame(ew_metrics_list)
    if len(ew_metrics) > 0:
        ew_metrics['weighting'] = 'Equal-Weighted'
    vw_returns = portfolio_data[['date', 'vw_return', group_col]].copy()
    vw_returns = vw_returns.rename(columns={'vw_return': 'portfolio_return'})
    vw_metrics_list = []
    for group_name, group_data in vw_returns.groupby(group_col):
        r = run_reg(group_data[['date', 'portfolio_return']], market_data, rf_rate)
        if r is not None and len(r) > 0:
            vw_metrics_list.append({**r.to_dict(), group_col: group_name})
    vw_metrics = pd.DataFrame(vw_metrics_list)
    if len(vw_metrics) > 0:
        vw_metrics['weighting'] = 'Value-Weighted'
    if len(ew_metrics) > 0 and len(vw_metrics) > 0:
        all_metrics = pd.concat([ew_metrics, vw_metrics], ignore_index=True)
    elif len(ew_metrics) > 0:
        all_metrics = ew_metrics
    elif len(vw_metrics) > 0:
        all_metrics = vw_metrics
    else:
        return pd.DataFrame()
    all_metrics['portfolio'] = all_metrics['weighting'] + '_' + all_metrics[group_col].astype(str)
    all_metrics = all_metrics.drop(columns=[group_col])
    return all_metrics

def format_performance_table(metrics, model='capm'):
    if metrics is None or (isinstance(metrics, pd.DataFrame) and (metrics.empty or 'portfolio' not in metrics.columns)):
        return metrics if isinstance(metrics, pd.DataFrame) else pd.DataFrame()
    rounding_map = {
        'arithmetic_monthly_return': 6, 'annualized_return': 4, 'alpha': 6, 'beta': 6,
        'alpha_tstat': 6, 'beta_tstat': 2, 'alpha_lower_ci': 6, 'alpha_upper_ci': 6,
        'beta_lower_ci': 6, 'beta_upper_ci': 6, 'r_squared': 6, 'sharpe_ratio': 6,
        'monthly_vol': 6, 'annualized_vol': 6,
        'beta_mkt': 6, 'beta_smb': 6, 'beta_hml': 6, 'beta_smb_tstat': 2, 'beta_hml_tstat': 2
    }
    table = metrics.copy()
    if 'portfolio' not in table.columns and 'weighting' in table.columns:
        group_col = [c for c in ['portfolio_id', 'research_not'] if c in table.columns]
        if group_col:
            table['portfolio'] = table['weighting'] + '_' + table[group_col[0]].astype(str)
    table['Portfolio'] = table['portfolio']
    table = format_numeric_cols(table, rounding_map)
    table['Alpha p-value'] = table['alpha_pval'].apply(lambda x: f"{x:.3e}")
    table['Beta p-value'] = table['beta_pval'].apply(lambda x: f"{x:.3e}")
    base_cols = ['Portfolio', 'arithmetic_monthly_return', 'monthly_vol', 'annualized_return', 'alpha', 'beta', 'alpha_tstat', 'beta_tstat', 'Alpha p-value', 'Beta p-value', 'alpha_lower_ci', 'alpha_upper_ci', 'beta_lower_ci', 'beta_upper_ci', 'r_squared', 'sharpe_ratio']
    renames = {'arithmetic_monthly_return': 'Mean Return (monthly)', 'monthly_vol': 'Std Return (monthly)', 'annualized_return': 'Annualized Return', 'alpha': 'Alpha (Monthly)', 'beta': 'Beta (Mkt)', 'alpha_tstat': 'Alpha t-stat', 'beta_tstat': 'Beta t-stat', 'alpha_lower_ci': 'Alpha Lower CI (95%)', 'alpha_upper_ci': 'Alpha Upper CI (95%)', 'beta_lower_ci': 'Beta Lower CI (95%)', 'beta_upper_ci': 'Beta Upper CI (95%)', 'r_squared': 'R-squared', 'sharpe_ratio': 'Sharpe Ratio'}
    if model == 'ff3' and 'beta_smb' in table.columns:
        base_cols = base_cols[:5] + ['beta_smb', 'beta_hml', 'beta_smb_tstat', 'beta_hml_tstat'] + base_cols[5:]
        renames['beta_smb'], renames['beta_hml'] = 'SMB Beta', 'HML Beta'
        renames['beta_smb_tstat'], renames['beta_hml_tstat'] = 'SMB t-stat', 'HML t-stat'
    out = table[[c for c in base_cols if c in table.columns]].copy()
    return out.rename(columns=renames)

def create_alpha_table(portfolio_metrics, market_returns, rf_rate_monthly):
    benchmark_sharpe_vw = calculate_sharpe_ratio(market_returns['vwretd'], rf_rate_monthly)
    benchmark_rows = pd.DataFrame({
        'Portfolio': ['CRSP VW Benchmark'],
        'Alpha (monthly %)': ['—'], 't-statistic': ['—'], 'p-value': ['—'],
        'Beta': ['—'], 'R²': ['—'],
        'Sharpe': [round(benchmark_sharpe_vw, 2)],
        'is_significant': [False], 'sort_order': [99]
    })
    def portfolio_display_name(x):
        # portfolio is e.g. 'Value-Weighted_No R&D', 'Equal-Weighted_Q3'
        parts = x.split('_', 1)
        if len(parts) != 2:
            return x
        wname = 'VW' if 'Value' in parts[0] else 'EW'
        return f"{parts[1]} {wname}"
    alpha_table = portfolio_metrics.copy()
    alpha_table['Portfolio'] = alpha_table['portfolio'].apply(portfolio_display_name)
    alpha_table['alpha_monthly_pct'] = (alpha_table['alpha'] * 100).round(2)
    alpha_table['alpha_star'] = alpha_table['alpha_pval'].apply(lambda x: '***' if x < 0.01 else '**' if x < 0.05 else '*' if x < 0.10 else '')
    alpha_table['Alpha (monthly %)'] = alpha_table['alpha_monthly_pct'].apply(lambda x: f"{x:.2f}") + alpha_table['alpha_star']
    alpha_table['t-statistic'] = alpha_table['alpha_tstat'].round(2)
    alpha_table['p-value'] = alpha_table['alpha_pval'].apply(lambda x: f"{x:.3f}")
    alpha_table['Beta'] = alpha_table['beta'].round(2)
    is_ff3 = 'beta_smb' in alpha_table.columns and 'beta_hml' in alpha_table.columns
    if is_ff3:
        alpha_table['Beta (Mkt)'] = alpha_table['beta'].round(2)
        alpha_table['Beta (SMB)'] = alpha_table['beta_smb'].round(2)
        alpha_table['Beta (HML)'] = alpha_table['beta_hml'].round(2)
    alpha_table['R²'] = alpha_table['r_squared'].round(2)
    alpha_table['Sharpe'] = alpha_table['sharpe_ratio'].round(2)
    alpha_table['is_significant'] = alpha_table['alpha_pval'] < 0.10
    sort_map = {'No R&D VW': 0, 'Q1 VW': 1, 'Q2 VW': 2, 'Q3 VW': 3, 'Q4 VW': 4, 'Q5 VW': 5, 'Q5-No R&D VW': 5.3, 'Q5-Q1 VW': 5.5,
                'No R&D EW': 6, 'Q1 EW': 7, 'Q2 EW': 8, 'Q3 EW': 9, 'Q4 EW': 10, 'Q5 EW': 11, 'Q5-No R&D EW': 11.3, 'Q5-Q1 EW': 11.5}
    alpha_table['sort_order'] = alpha_table['Portfolio'].map(sort_map).fillna(9)
    alpha_table = alpha_table.sort_values('sort_order')
    display_cols = ['Portfolio', 'Alpha (monthly %)', 't-statistic', 'p-value']
    if is_ff3:
        display_cols += ['Beta (Mkt)', 'Beta (SMB)', 'Beta (HML)']
    else:
        display_cols += ['Beta']
    display_cols += ['R²', 'Sharpe', 'is_significant', 'sort_order']
    alpha_table = alpha_table[[c for c in display_cols if c in alpha_table.columns]].astype(str)
    if is_ff3:
        benchmark_rows = pd.DataFrame({
            'Portfolio': ['CRSP VW Benchmark'], 'Alpha (monthly %)': ['—'], 't-statistic': ['—'], 'p-value': ['—'],
            'Beta (Mkt)': ['—'], 'Beta (SMB)': ['—'], 'Beta (HML)': ['—'], 'R²': ['—'],
            'Sharpe': [round(benchmark_sharpe_vw, 2)], 'is_significant': [False], 'sort_order': [99]
        })
    benchmark_rows = benchmark_rows.astype(str)
    alpha_table = pd.concat([alpha_table, benchmark_rows], ignore_index=True)
    alpha_table = alpha_table.sort_values('sort_order').drop(columns='sort_order')
    return alpha_table

def create_alpha_table_html(alpha_table, custom_css, caption_suffix='', table_type=""):
    """Create HTML alpha table with same styling (color-coded rows). Columns taken from alpha_table."""
    alpha_table_display = alpha_table.drop(columns='is_significant', errors='ignore')
    display_cols = [c for c in alpha_table_display.columns]
    if table_type == 'FF3':
        caption_title = "Alpha Estimation Results: Three-Factor Model"
        caption_eq = "Rp - Rƒ = α + β_mkt(MKT-Rƒ) + β_smb SMB + β_hml HML"
    else:
        caption_title = "Alpha Estimation Results: Single-Factor Model"
        caption_eq = "Rp - Rƒ = α + β(R_CRSP_VW - Rƒ)"
    html = f"""
    <!DOCTYPE html>
    <html>
    <head>
    {custom_css}
    </head>
    <body>
    <table class="table table-dark table-striped">
    <caption><b>{caption_title}</b><br>
    <i>{caption_eq}</i>{caption_suffix}<br>
    <i>{table_type}</i></caption>
    <thead>
    <tr>
    """ + "".join(f"<th>{c}</th>" for c in display_cols) + """
    </tr>
    </thead>
    <tbody>
    """
    for _, row in alpha_table_display.iterrows():
        portfolio = row['Portfolio']
        if portfolio == 'CRSP VW Benchmark':
            color = 'white'
        elif 'No R&D' in portfolio:
            color = '#003f5c'
        elif 'Q5-No R&D' in portfolio: color = '#00b4d8'
        elif 'Q5-Q1' in portfolio: color = '#00c9a7'
        elif 'Q1' in portfolio: color = '#ffa600'
        elif 'Q2' in portfolio: color = '#dd5182'
        elif 'Q3' in portfolio: color = '#955196'
        elif 'Q4' in portfolio: color = '#bc5090'
        elif 'Q5' in portfolio: color = '#ff6361'
        else:
            color = 'white'
        bold = 'font-weight: bold;' if portfolio == 'CRSP VW Benchmark' or 'No R&D' in portfolio or 'Q5-Q1' in portfolio or any(f'Q{i}' in portfolio for i in range(1,6)) else ''
        cells = "".join(f"<td>{row[c]}</td>" for c in display_cols)
        html += f"""
        <tr style="color: {color}; {bold}">
        {cells}
        </tr>
        """
    html += """
    </tbody>
    </table>
    </body>
    </html>
    """
    return html

custom_css = """
<style>
@import url('https://fonts.googleapis.com/css2?family=Tomorrow:wght@400;600;700&display=swap');
body { font-family: 'Tomorrow', sans-serif; background-color: #1d1d1c; padding: 1rem; margin: 0; color: white; }
.table-dark.table-striped tbody tr:nth-of-type(odd) { background-color: rgba(255, 255, 255, 0.05) !important; }
.table-dark.table-striped tbody tr:nth-of-type(even) { background-color: rgba(0, 0, 0, 0.15) !important; }
.table-dark tbody tr[style*='color'] { background-color: inherit !important; }
.table-dark td, .table-dark th { padding: 0.5rem 0.75rem !important; }
</style>
"""

def create_performance_chart(chart_data, panel_type='vw', colors=None):
    pattern = 'vw_' if panel_type == 'vw' else 'ew_'
    benchmark_name = 'CRSP VW Benchmark'
    panel_data = chart_data[chart_data['key'].str.contains(pattern, na=False)].copy()
    def key_to_name(x):
        if 'CRSP' in str(x): return benchmark_name
        if '_' in str(x): return str(x).split('_', 1)[-1]
        return x
    panel_data['portfolio_name'] = panel_data['key'].apply(key_to_name)
    fig = go.Figure()
    for portfolio in panel_data['portfolio_name'].unique():
        port_data = panel_data[panel_data['portfolio_name'] == portfolio].sort_values('date')
        color = (colors or {}).get(portfolio, '#888')
        fig.add_trace(go.Scatter(x=port_data['date'], y=port_data['cumvalue'], mode='lines', name=portfolio, line=dict(color=color, width=3)))
    fig.update_layout(title='Cumulative Performance (1980-2022)', xaxis_title='Date', yaxis_title='Cumulative Value ($)',
                      yaxis=dict(rangemode='tozero', tickformat='$.2f'), template='plotly_dark', height=500)
    return fig


## Build Market Returns and Portfolio Metrics

In [23]:
mkt['date'] = pd.to_datetime(mkt['date']).dt.to_period('M').dt.to_timestamp()
ff['date'] = pd.to_datetime(ff['date']).dt.to_period('M').dt.to_timestamp()
for c in ['rf', 'mktrf', 'smb', 'hml']:
    ff[c] = pd.to_numeric(ff[c], errors='coerce')
market_returns = mkt.merge(ff, on='date', how='left')
if 'rf' not in market_returns.columns or market_returns['rf'].isna().all():
    raise ValueError("Risk-free rate 'rf' must be present (from ff.factors_monthly).")
market_returns['vwretd'] = pd.to_numeric(market_returns['vwretd'], errors='coerce')
market_returns['ewretd'] = pd.to_numeric(market_returns['ewretd'], errors='coerce')

rf_rate_monthly = market_returns['rf'].mean()

def run_quintile_metrics(analysis_data, market_returns, rf_rate_monthly):
    """Portfolio returns (with Q5-Q1 spread) and CAPM/FF3 metrics for one analysis_data."""
    pr = calculate_portfolio_returns(analysis_data, 'portfolio_id')
    pr = add_q5_minus_q1_spread(pr, 'portfolio_id')
    pm_capm = calculate_all_portfolio_metrics(pr, market_returns, 'R&D Quintiles', rf_rate_monthly, model='capm')
    pm_ff3 = calculate_all_portfolio_metrics(pr, market_returns, 'R&D Quintiles', rf_rate_monthly, model='ff3')
    return pr, pm_capm, pm_ff3

def metrics_for_period(portfolio_returns, market_returns, rf_rate_monthly, start_date, end_date):
    """Compute CAPM and FF3 metrics for a subperiod (start_date, end_date inclusive)."""
    start_ts = pd.Timestamp(start_date)
    end_ts = pd.Timestamp(end_date)
    pr = portfolio_returns[(portfolio_returns['date'] >= start_ts) & (portfolio_returns['date'] <= end_ts)].copy()
    mr = market_returns[(market_returns['date'] >= start_ts) & (market_returns['date'] <= end_ts)].copy()
    if pr.empty or mr.empty:
        return None, None
    pm_capm = calculate_all_portfolio_metrics(pr, mr, 'R&D Quintiles', rf_rate_monthly, model='capm')
    pm_ff3 = calculate_all_portfolio_metrics(pr, mr, 'R&D Quintiles', rf_rate_monthly, model='ff3')
    return pm_capm, pm_ff3

SUBPERIODS = [
    ('1980-2000', '1980-01-01', '2000-12-31'),
    ('2001-2019', '2001-01-01', '2019-12-31'),
]

portfolio_returns_norm, portfolio_metrics_capm_norm, portfolio_metrics_ff3_norm = run_quintile_metrics(analysis_data_norm, market_returns, rf_rate_monthly)
portfolio_returns_nonnorm, portfolio_metrics_capm_nonnorm, portfolio_metrics_ff3_nonnorm = run_quintile_metrics(analysis_data_nonnorm, market_returns, rf_rate_monthly)

/var/folders/fj/plbn4w9d6fs55651zh_pxsd80000gn/T/ipykernel_50644/3159165517.py:24: DeprecationWarning: DataFrameGroupBy.apply operated on the grouping columns. This behavior is deprecated, and in a future version of pandas the grouping columns will be excluded from the operation. Either pass `include_groups=False` to exclude the groupings or explicitly select the grouping columns after groupby to silence this warning.
  portfolio_returns = data.groupby(['date', group_var]).apply(


NameError: name 'add_q5_minus_q1_spread' is not defined

In [ ]:
portfolio_returns.sort_values('date')

,date,portfolio_id,vw_return,ew_return,total_mktcap,n_stocks
0,1980-02-29,No R&D,0.002692,0.032188,9.976506e+06,129.0
3126,1980-02-29,Q5-No R&D,-0.037479,-0.041025,0.000000e+00,0.0
2605,1980-02-29,Q5,-0.034787,-0.008837,1.488507e+07,21.0
2084,1980-02-29,Q4,-0.011648,-0.003387,1.819646e+06,20.0
1563,1980-02-29,Q3,0.030069,0.026527,7.290362e+05,22.0
...,...,...,...,...,...,...
1562,2023-06-30,Q2,0.114034,0.083219,2.999126e+07,46.0
1041,2023-06-30,Q1,0.083224,0.051847,1.279424e+07,47.0
520,2023-06-30,No R&D,0.067414,0.067341,9.618002e+08,160.0
3646,2023-06-30,Q5-No R&D,-0.014646,-0.000371,0.000000e+00,0.0


## Outputs: Performance Tables, Alpha Table, Chart

In [ ]:
from IPython.display import HTML, display

TEST_PERIOD = '1980–2022'
colors = {'CRSP VW Benchmark': '#58508d', 'No R&D': '#003f5c', 'Q1': '#ffa600', 'Q2': '#dd5182', 'Q3': '#955196', 'Q4': '#bc5090', 'Q5': '#ff6361', 'Q5-No R&D': '#00b4d8', 'Q5-Q1': '#00c9a7'}
market_vw = market_returns.copy()
market_vw['date'] = market_vw['date'] + pd.offsets.MonthEnd(0)
market_vw['key'] = 'vw_CRSP_VW_Benchmark'
market_vw['value'] = market_vw['vwretd']

variants = [
    ('Normalized (XRD % of net income)', portfolio_returns_norm, portfolio_metrics_capm_norm, portfolio_metrics_ff3_norm),
    ('Non-normalized (raw XRD)', portfolio_returns_nonnorm, portfolio_metrics_capm_nonnorm, portfolio_metrics_ff3_nonnorm),
]

returns_data_norm = None
returns_data_nonnorm = None

for rd_label, portfolio_returns, portfolio_metrics_capm, portfolio_metrics_ff3 in variants:
    print('\n' + '#'*70)
    print(f'# {rd_label}')
    print('#'*70)
    print('Test period:', TEST_PERIOD, '| 6 portfolios: No R&D + Q1–Q5 (equal count)')

    perf_capm = format_performance_table(portfolio_metrics_capm, model='capm')
    alpha_capm = create_alpha_table(portfolio_metrics_capm, market_returns, rf_rate_monthly)
    print('\n' + '='*70)
    print('CAPM')
    print('='*70)
    print('\nPerformance Table (CAPM):')
    print(perf_capm.to_string())
    print('\nAlpha Estimation Results (CAPM):')
    display(HTML(create_alpha_table_html(alpha_capm, custom_css, caption_suffix=f'<br>{rd_label}<br>Test period: {TEST_PERIOD}', table_type='CAPM')))

    perf_ff3 = format_performance_table(portfolio_metrics_ff3, model='ff3')
    alpha_ff3 = create_alpha_table(portfolio_metrics_ff3, market_returns, rf_rate_monthly)
    print('\n' + '='*70)
    print('Fama-French 3-factor')
    print('='*70)
    print('\nPerformance Table (FF3):')
    print(perf_ff3.to_string())
    print('\nAlpha Estimation Results (FF3):')
    display(HTML(create_alpha_table_html(alpha_ff3, custom_css, caption_suffix=f'<br>{rd_label}<br>Test period: {TEST_PERIOD}', table_type='FF3')))

    # Subperiod tests: 1980-2000 and 2001-2019
    for period_label, start_date, end_date in SUBPERIODS:
        pm_capm_sub, pm_ff3_sub = metrics_for_period(portfolio_returns, market_returns, rf_rate_monthly, start_date, end_date)
        if pm_capm_sub is None or pm_capm_sub.empty:
            continue
        print('\n' + '-'*70)
        print(f'Subperiod: {period_label}')
        print('-'*70)
        perf_capm_sub = format_performance_table(pm_capm_sub, model='capm')
        alpha_capm_sub = create_alpha_table(pm_capm_sub, market_returns, rf_rate_monthly)
        print('\nCAPM:')
        print(perf_capm_sub.to_string())
        display(HTML(create_alpha_table_html(alpha_capm_sub, custom_css, caption_suffix=f'<br>{rd_label}<br>Test period: {period_label}', table_type='CAPM')))
        perf_ff3_sub = format_performance_table(pm_ff3_sub, model='ff3')
        alpha_ff3_sub = create_alpha_table(pm_ff3_sub, market_returns, rf_rate_monthly)
        print('\nFF3:')
        print(perf_ff3_sub.to_string())
        display(HTML(create_alpha_table_html(alpha_ff3_sub, custom_css, caption_suffix=f'<br>{rd_label}<br>Test period: {period_label}', table_type='FF3')))

    chart_wide = portfolio_returns.copy()
    chart_wide['key'] = chart_wide['portfolio_id'].astype(str)
    chart_data = chart_wide.melt(id_vars=['date', 'portfolio_id', 'key'], value_vars=['ew_return', 'vw_return'], var_name='weighting', value_name='value')
    chart_data['key'] = chart_data['weighting'].str.replace('_return', '') + '_' + chart_data['portfolio_id'].astype(str)
    chart_data = pd.concat([chart_data[['date', 'key', 'value']], market_vw[['date', 'key', 'value']]], ignore_index=True)
    common_start = chart_data.groupby('key')['date'].min().max()
    chart_data = chart_data[chart_data['date'] >= common_start].copy()
    chart_data = chart_data.sort_values(['key', 'date'])
    chart_data['cumvalue'] = (1.0 + pd.to_numeric(chart_data['value'], errors='coerce')).groupby(chart_data['key']).cumprod()
    chart_data['cumvalue'] = chart_data.groupby('key')['cumvalue'].transform(lambda x: x / x.iloc[0])
    fig_vw = create_performance_chart(chart_data, panel_type='vw', colors=colors)
    fig_vw.update_layout(title=f'Cumulative Performance (1980-2022) — {rd_label}')
    fig_vw.show()

    wide_ew = portfolio_returns.pivot(index='date', columns='portfolio_id', values='ew_return')
    wide_vw = portfolio_returns.pivot(index='date', columns='portfolio_id', values='vw_return')
    wide_ew = wide_ew.add_suffix('_ew')
    wide_vw = wide_vw.add_suffix('_vw')
    returns_data = wide_ew.join(wide_vw).reset_index()
    if 'Normalized' in rd_label:
        returns_data_norm = returns_data
    else:
        returns_data_nonnorm = returns_data

db.close()


######################################################################
# Normalized (XRD % of net income)
######################################################################
Test period: 1980–2022 | 6 portfolios: No R&D + Q1–Q5 (equal count)

CAPM

Performance Table (CAPM):
                Portfolio  Mean Return (monthly)  Std Return (monthly)  Annualized Return  Alpha (Monthly)  Beta (Mkt)  Alpha t-stat  Beta t-stat Alpha p-value Beta p-value  Alpha Lower CI (95%)  Alpha Upper CI (95%)  Beta Lower CI (95%)  Beta Upper CI (95%)  R-squared  Sharpe Ratio
0   Equal-Weighted_No R&D               0.011025              0.064860             0.1406        -0.000013    1.191028     -0.008240        34.00     9.934e-01   3.611e-134             -0.003156              0.003129             1.122219             1.259838   0.690206      0.415364
1       Equal-Weighted_Q1               0.010419              0.051115             0.1324         0.000579    1.007757      0.565614        44.95     5.7

Portfolio,Alpha (monthly %),t-statistic,p-value,Beta,R²,Sharpe
No R&D VW,-0.02,-0.44,0.662,1.0,0.96,0.48
Q1 VW,0.06,0.56,0.578,0.81,0.73,0.47
Q4 EW,0.33**,2.2,0.028,1.24,0.74,0.6
Q5 EW,0.55***,2.91,0.004,1.28,0.65,0.67
Q5-Q1 EW,0.17,1.01,0.312,0.28,0.1,0.31
Q2 VW,0.20*,1.93,0.054,1.06,0.81,0.58
Q3 VW,0.04,0.35,0.727,1.09,0.81,0.47
Q4 VW,0.05,0.36,0.722,1.21,0.77,0.47
Q5 VW,0.13,0.76,0.447,1.32,0.72,0.49
Q5-Q1 VW,-0.26,-1.14,0.255,0.51,0.17,0.05



Fama-French 3-factor

Performance Table (FF3):
                Portfolio  Mean Return (monthly)  Annualized Return  Alpha (Monthly)  SMB Beta  HML Beta  SMB t-stat  HML t-stat  Beta (Mkt)  Alpha t-stat  Beta t-stat Alpha p-value Beta p-value  Alpha Lower CI (95%)  Alpha Upper CI (95%)  Beta Lower CI (95%)  Beta Upper CI (95%)  R-squared  Sharpe Ratio
0   Equal-Weighted_No R&D               0.011025             0.1406        -0.000601  0.985974  0.184255       26.51        5.29    1.057652     -0.560064        43.59     5.757e-01   2.822e-175             -0.002708              0.001506             1.009985             1.105318   0.863009      0.415364
1       Equal-Weighted_Q1               0.010419             0.1324        -0.000474  0.548335  0.296368       20.94       12.10    0.959727     -0.628067        56.20     5.302e-01   2.457e-222             -0.001957              0.001009             0.926177             0.993277   0.890725      0.485963
2       Equal-Weighted_Q2         

Portfolio,Alpha (monthly %),t-statistic,p-value,Beta (Mkt),Beta (SMB),Beta (HML),R²,Sharpe
No R&D VW,-0.06,-1.47,0.141,0.98,0.13,0.03,0.96,0.48
Q1 VW,-0.02,-0.25,0.802,0.86,-0.26,0.13,0.76,0.47
Q4 EW,0.34***,3.55,0.000,1.09,0.91,-0.07,0.89,0.6
Q5 EW,0.61***,5.01,0.000,1.07,1.12,-0.19,0.86,0.67
Q5-Q1 EW,0.33***,2.59,0.010,0.12,0.58,-0.49,0.48,0.31
Q2 VW,0.21**,2.13,0.034,1.05,-0.06,-0.18,0.83,0.58
Q3 VW,0.07,0.72,0.471,1.06,-0.06,-0.26,0.84,0.47
Q4 VW,0.11,0.95,0.344,1.13,0.21,-0.34,0.81,0.47
Q5 VW,0.23,1.59,0.113,1.2,0.35,-0.46,0.79,0.49
Q5-Q1 VW,-0.08,-0.4,0.688,0.34,0.61,-0.59,0.41,0.05



----------------------------------------------------------------------
Subperiod: 1980-2000
----------------------------------------------------------------------

CAPM:
                Portfolio  Mean Return (monthly)  Std Return (monthly)  Annualized Return  Alpha (Monthly)  Beta (Mkt)  Alpha t-stat  Beta t-stat Alpha p-value Beta p-value  Alpha Lower CI (95%)  Alpha Upper CI (95%)  Beta Lower CI (95%)  Beta Upper CI (95%)  R-squared  Sharpe Ratio
0   Equal-Weighted_No R&D               0.011485              0.057271             0.1469        -0.001456    1.022586     -0.660381        21.00     5.096e-01    5.002e-57             -0.005798              0.002886             0.926668             1.118505   0.639068      0.360611
1       Equal-Weighted_Q1               0.010958              0.047847             0.1397        -0.001396    0.941672     -0.963146        29.41     3.364e-01    5.682e-83             -0.004251              0.001459             0.878607             1.004738   

...

Portfolio,Alpha (monthly %),t-statistic,p-value,Beta (Mkt),Beta (SMB),Beta (HML),R²,Sharpe
No R&D VW,-0.15**,-2.09,0.037,0.98,0.04,0.11,0.94,0.51
Q1 VW,-1.00***,-5.74,0.000,1.05,0.92,-0.03,0.85,-0.15
Q4 EW,0.36**,2.58,0.010,1.04,1.11,-0.17,0.91,0.49
Q5 EW,0.17,1.35,0.179,1.13,0.54,-0.03,0.9,0.56
Q5-Q1 EW,-0.56**,-2.08,0.038,0.25,-0.65,-0.15,0.23,-0.28
Q2 VW,-0.26**,-2.16,0.031,1.05,0.9,-0.18,0.92,0.21
Q3 VW,0.14,1.16,0.249,1.05,0.84,-0.44,0.93,0.35
Q4 VW,0.08,0.61,0.545,1.06,0.59,-0.33,0.91,0.38
Q5 VW,0.13,1.38,0.169,0.95,-0.08,-0.23,0.92,0.55
Q5-Q1 VW,0.57***,2.78,0.006,-0.09,-1.0,-0.19,0.52,0.39



----------------------------------------------------------------------
Subperiod: 2001-2019
----------------------------------------------------------------------

CAPM:
                Portfolio  Mean Return (monthly)  Std Return (monthly)  Annualized Return  Alpha (Monthly)  Beta (Mkt)  Alpha t-stat  Beta t-stat Alpha p-value Beta p-value  Alpha Lower CI (95%)  Alpha Upper CI (95%)  Beta Lower CI (95%)  Beta Upper CI (95%)  R-squared  Sharpe Ratio
0   Equal-Weighted_No R&D               0.010133              0.060157             0.1286         0.002345    1.225054      1.203954        27.18     2.299e-01    3.643e-73             -0.001493              0.006184             1.136234             1.313874   0.765722      0.516363
1       Equal-Weighted_Q1               0.009615              0.066160             0.1217         0.002107    1.173225      0.735121        17.69     4.630e-01    1.565e-44             -0.003541              0.007755             1.042531             1.303919   

Portfolio,Alpha (monthly %),t-statistic,p-value,Beta,R²,Sharpe
No R&D VW,0.06,0.82,0.415,0.93,0.93,0.47
Q1 VW,-0.25,-1.34,0.181,1.19,0.77,0.23
Q4 EW,0.37,1.52,0.131,1.56,0.77,0.55
Q5 EW,0.33*,1.75,0.081,1.47,0.84,0.56
Q5-Q1 EW,-0.00,-0.01,0.993,0.3,0.11,0.14
Q2 VW,0.18,0.92,0.358,1.24,0.77,0.48
Q3 VW,0.14,0.98,0.330,1.25,0.86,0.49
Q4 VW,-0.00,-0.0,0.997,1.28,0.87,0.41
Q5 VW,0.08,0.87,0.387,1.06,0.92,0.47
Q5-Q1 VW,0.21,0.93,0.352,-0.13,0.03,0.14



FF3:
                Portfolio  Mean Return (monthly)  Annualized Return  Alpha (Monthly)  SMB Beta  HML Beta  SMB t-stat  HML t-stat  Beta (Mkt)  Alpha t-stat  Beta t-stat Alpha p-value Beta p-value  Alpha Lower CI (95%)  Alpha Upper CI (95%)  Beta Lower CI (95%)  Beta Upper CI (95%)  R-squared  Sharpe Ratio
0   Equal-Weighted_No R&D               0.010133             0.1286         0.000836  0.870920  0.255565       14.84        4.92    1.053354      0.595861        30.73     5.519e-01    2.642e-82             -0.001928              0.003600             0.985815             1.120893   0.880202      0.516363
1       Equal-Weighted_Q1               0.009615             0.1217         0.000633  0.999541 -0.041035        9.65       -0.45    0.985151      0.255811        16.28     7.983e-01    7.416e-40             -0.004246              0.005512             0.865923             1.104378   0.691349      0.442370
2       Equal-Weighted_Q2               0.011290             0.1442         

Portfolio,Alpha (monthly %),t-statistic,p-value,Beta (Mkt),Beta (SMB),Beta (HML),R²,Sharpe
No R&D VW,0.02,0.21,0.832,0.91,0.08,0.05,0.93,0.47
Q1 VW,-0.39***,-2.82,0.005,1.04,0.8,0.16,0.87,0.23
Q4 EW,0.20,1.2,0.231,1.38,1.08,-0.3,0.89,0.55
Q5 EW,0.22,1.34,0.183,1.38,0.59,-0.19,0.88,0.56
Q5-Q1 EW,0.03,0.14,0.889,0.4,-0.41,-0.16,0.19,0.14
Q2 VW,0.06,0.38,0.704,1.12,0.73,-0.18,0.85,0.48
Q3 VW,0.03,0.31,0.755,1.13,0.71,-0.2,0.93,0.49
Q4 VW,-0.08,-0.72,0.473,1.22,0.45,-0.37,0.93,0.41
Q5 VW,0.06,0.82,0.415,1.08,-0.04,-0.29,0.95,0.47
Q5-Q1 VW,0.33**,2.02,0.044,0.05,-0.84,-0.45,0.5,0.14
